## Basic imports

In [59]:
# importing necessary libraries
import numpy as np #for numeric operations
import pandas as pd # for data manipulation and analysis
import matplotlib.pyplot as plt # data visualization
%matplotlib inline

# importing WordCloud for text visualization
from wordcloud import WordCloud

# importing NLTK for natural language processing
import nltk
from nltk.corpus import stopwords # removing stopwords

# downloading nltk data: stopwords and tokenizer data
nltk.download("stopwords")
nltk.download("punkt")

[nltk_data] Downloading package stopwords to /home/om/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/om/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## 1. DataIngestion(Loading the data from the file)

In [60]:
df= pd.read_csv("spam.csv")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [61]:
df.columns

Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')

## 2. DataPreprocessing

### Dropping unnecessary columns

In [62]:
columns_drop=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']
def drop_columns(dataFrame:pd.DataFrame,columns_list:list)->pd.DataFrame:
    """
    This function is used to drop the given columns from the given dataframe.
    """
    dataFrame.drop(columns=columns_list,inplace=True)
    return dataFrame

df=drop_columns(df,columns_drop)
df

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


### function to rename the column names

In [63]:
columns_old_names=['v1', 'v2']
columns_new_names=['target','text']
def rename_columns(dataFrame:pd.DataFrame,columns_old_names:list,columns_new_names:list)->pd.DataFrame:
    """ 
    This function takes the dataFrame, columns_old_names list and rename the columns using columns_new_names list.
    """
    for i in range(len(columns_old_names)):
        dataFrame.rename(columns={columns_old_names[i]:columns_new_names[i]},inplace=True)
    return dataFrame

df=rename_columns(df,columns_old_names,columns_new_names)  
df  

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


### checking the categories in target column

In [64]:
df["target"].unique()

array(['ham', 'spam'], dtype=object)

### function to convert the 'target' column to numeric values

In [65]:
from sklearn.preprocessing import LabelEncoder
def converting_text_to_labels(dataFrame:pd.DataFrame,column_name:str)->pd.DataFrame:
    """ 
    This function converts the text data of a data frame to numbers using LabelEncoder.
    """
    le=LabelEncoder()
    dataFrame[column_name]= le.fit_transform(dataFrame[column_name])
    print("Categorical classes: ",le.classes_)
    return dataFrame

df= converting_text_to_labels(df,"target")
df

Categorical classes:  ['ham' 'spam']


,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will Ì_ b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...


### function to remove the duplicates

In [66]:

def remove_duplicates(dataFrame:pd.DataFrame)->pd.DataFrame:
    """ 
    This function is used to remove the duplicates from the input dataframe.
    """
    print("Total rows before removing duplicates: ",len(df))
    print("Total_duplicates: ",df.duplicated().sum())
    df.drop_duplicates(inplace=True)
    print("Total rows after removing duplicates: ",len(df))
    return df

df= remove_duplicates(df)
df.info()

Total rows before removing duplicates:  5572
Total_duplicates:  403
Total rows after removing duplicates:  5169
<class 'pandas.core.frame.DataFrame'>
Index: 5169 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   target  5169 non-null   int64 
 1   text    5169 non-null   object
dtypes: int64(1), object(1)
memory usage: 121.1+ KB


### function to remove null values

In [67]:
def remove_null(dataFrame:pd.DataFrame)->pd.DataFrame:
    """ 
    This function is used to remove the null values from the input dataframe.
    """
    print("Total rows before removing null values: ",len(df))
    df.dropna(inplace=True)
    print("Total rows after removing null values: ",len(df))
    return df

df= remove_null(df)
df.isnull().sum()

Total rows before removing null values:  5169
Total rows after removing null values:  5169


target    0
text      0
dtype: int64

In [68]:
import string
print(string.punctuation)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


## 3. feature Engineering

### function to transform the text

In [69]:
# importing porter stemmer for text stemming
from nltk.stem.porter import PorterStemmer
# importing the string module to handle special characters
import string

nltk.download('punkt_tab') # to run nltk.word_tokenize(text1)

def transform_text(text:str)->str:
    """ 
    This function is used to transform the given text using the in-built methods and a list (y).
    """
    #1. converting the text to lowercase
    text1=text.lower()

    #2. tokenization using nltk
    text1= nltk.word_tokenize(text1)

    #3. removing special characters
    y=[]
    for i in text1:
        if i.isalnum():
            y.append(i)
    text1=y[:]  
    y.clear()

    #4. removing stopwords and punctuations
    for i in text1:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)
    text1=y[:]
    y.clear()

    #5.stemming using porter stemmer
    ps= PorterStemmer()  
    for i in text1:
        y.append(ps.stem(i))  
    text1=y[:]
    y.clear()

    return " ".join(text1)

transform_text('Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...')

[nltk_data] Downloading package punkt_tab to /home/om/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


'go jurong point crazi avail bugi n great world la e buffet cine got amor wat'

In [70]:
df["transformed_text"]=df["text"].apply(transform_text)
df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


### transforming the textual data to numerical using TFIDF or CounterVectorizer (BOW)

In [74]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
tfidf= TfidfVectorizer(max_features=500)
X= tfidf.fit_transform(df["transformed_text"]).toarray()
X

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(5169, 500))

In [75]:
y=df["target"]
y

0       0
1       0
2       1
3       0
4       0
       ..
5567    1
5568    0
5569    0
5570    0
5571    0
Name: target, Length: 5169, dtype: int64

### Train Test Split

In [76]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

## 4. model training & Evaluation

In [78]:
# imports for models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier

from xgboost import XGBClassifier


clfs={
    "SVC":SVC(kernel='sigmoid',gamma=1.0),
    "KNN":KNeighborsClassifier(),
    "NB":MultinomialNB(),
    "DT":DecisionTreeClassifier(max_depth=5),
    "LR":LogisticRegression(solver="liblinear",penalty='l1'),
    "RF":RandomForestClassifier(n_estimators=50,random_state=2),
    "AdaBoost":AdaBoostClassifier(n_estimators=50,random_state=2),
    "Bgc":BaggingClassifier(n_estimators=50,random_state=2),
    "ETC":ExtraTreesClassifier(n_estimators=50,random_state=2),
    "Gbdt":GradientBoostingClassifier(n_estimators=50,random_state=2),
    "Xgb":XGBClassifier(n_estimators=50,random_state=2)
}


In [79]:
from sklearn.metrics import accuracy_score,precision_score,f1_score,recall_score
def train_classifier(clf,X_train:np.array,y_train:np.array,X_test:np.array,y_test:np.array)->tuple:
    clf.fit(X_train,y_train)
    y_pred= clf.predict(X_test)

    accuracy= accuracy_score(y_test,y_pred)
    precision= precision_score(y_test,y_pred)
    f1= f1_score(y_test,y_pred)
    recall= recall_score(y_test,y_pred)

    return accuracy,precision,f1,recall


accuracy_scores=[]
precision_scores=[]
f1_scores=[]
recall_scores=[]

for name,model in clfs.items():
    accuracy,precision,f1,recall= train_classifier(model,X_train,y_train,X_test,y_test)
    print("For",name)
    print("accuracy,precision,f1,recall: ",accuracy," ",precision," ",f1," ",recall)

    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    f1_scores.append(f1)
    recall_scores.append(recall)

For SVC
accuracy,precision,f1,recall:  0.9695193434935522   0.955   0.880184331797235   0.8162393162393162
For KNN
accuracy,precision,f1,recall:  0.9173505275498242   0.9696969696969697   0.5765765765765766   0.41025641025641024
For NB
accuracy,precision,f1,recall:  0.9730363423212193   0.9653465346534653   0.8944954128440367   0.8333333333333334
For DT
accuracy,precision,f1,recall:  0.9337631887456037   0.8306010928961749   0.7290167865707434   0.6495726495726496
For LR
accuracy,precision,f1,recall:  0.9554513481828839   0.9114583333333334   0.8215962441314554   0.7478632478632479
For RF
accuracy,precision,f1,recall:  0.9730363423212193   0.9519230769230769   0.8959276018099548   0.8461538461538461
For AdaBoost
accuracy,precision,f1,recall:  0.9161781946072685   0.8053691275167785   0.6266318537859008   0.5128205128205128
For Bgc
accuracy,precision,f1,recall:  0.9607268464243846   0.8711111111111111   0.8540305010893247   0.8376068376068376
For ETC
accuracy,precision,f1,recall:  0.973

In [80]:
print(accuracy_scores)
print(precision_scores)
print(f1_scores)
print(recall_scores)

[0.9695193434935522, 0.9173505275498242, 0.9730363423212193, 0.9337631887456037, 0.9554513481828839, 0.9730363423212193, 0.9161781946072685, 0.9607268464243846, 0.9736225087924971, 0.9501758499413834, 0.9683470105509965]
[0.955, 0.9696969696969697, 0.9653465346534653, 0.8306010928961749, 0.9114583333333334, 0.9519230769230769, 0.8053691275167785, 0.8711111111111111, 0.9354838709677419, 0.9570552147239264, 0.9285714285714286]
[0.880184331797235, 0.5765765765765766, 0.8944954128440367, 0.7290167865707434, 0.8215962441314554, 0.8959276018099548, 0.6266318537859008, 0.8540305010893247, 0.9002217294900222, 0.7858942065491183, 0.8783783783783784]
[0.8162393162393162, 0.41025641025641024, 0.8333333333333334, 0.6495726495726496, 0.7478632478632479, 0.8461538461538461, 0.5128205128205128, 0.8376068376068376, 0.8675213675213675, 0.6666666666666666, 0.8333333333333334]
